# Urdu Orpheus: pronunciation comparison across adapters

Same benchmark prompts and seeds for every adapter, Whisper large-v3 CER/WER. Launch with `python kaggle_submit_stage2.py eval --adapter label=repo:run_id:checkpoint ...`.


In [ ]:
import os, sys, subprocess, re
from pathlib import Path

REPO_URL = "https://github.com/hassan-31x/orpheus-ft-aslp-urdutts.git"
REPO_REF = "PINNED_BY_PUSH_SCRIPT"  # Pinned commit; keep identical for every epoch and every account.
PROJECT_FOLDER = "."  # This standalone GitHub repository contains the scripts at its root.
HF_REPO_ID = "hassan-31x/orpheus-urdu-stage2-checkpoints"  # Private Hub MODEL repo holding every stage-2 epoch run.
HF_SECRET_NAME = "HF_TOKEN"  # Exact Kaggle secret LABEL used in your previous notebook; change if different.
EPOCH = 2  # 2 = from UrduTTS epoch 1; 3, 4, ... = from the finished previous stage-2 epoch.
LEARNING_RATE = None  # None uses the per-epoch value in stage2_plan.py. Keep fixed within one epoch.
TEST = True  # True: few-clip smoke run of this epoch. Set False after "TEST PASSED" for the full epoch.
TEST_TRAIN_CLIPS = 24
TEST_VALIDATION_CLIPS = 8
AUDIT_ONLY = False
EVAL_NAME = "PINNED_BY_PUSH_SCRIPT"  # Results go to evaluations/<EVAL_NAME>/ in HF_REPO_ID.
ADAPTERS = []  # label=repo:run_id:checkpoint entries, filled by kaggle_submit_stage2.py eval
REPEATS = 2
QUALITY_DEVICE = "cuda"
SESSION_HOURS = 11.75  # Kaggle kills sessions at 12 h; training saves + uploads a checkpoint before that.
CLONE = Path("/kaggle/working/orpheus-mixed-code")

if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", HF_REPO_ID):
    raise ValueError("HF_REPO_ID must be username/repository")
def load_hf_token(secret_name="HF_TOKEN", environ=None, secret_client_factory=None):
    # Keep a valid explicit environment token; do not overwrite it with a failed Kaggle lookup.
    environ = os.environ if environ is None else environ
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        value = environ.get(name, "").strip()
        if value:
            environ["HF_TOKEN"] = value
            return "environment:" + name
    for _path in sorted(Path("/kaggle/input").glob("**/hf_token.txt")):
        value = _path.read_text().strip()
        if value:
            environ["HF_TOKEN"] = value
            return "attached private Kaggle dataset"
    if not isinstance(secret_name, str) or not secret_name.strip():
        raise RuntimeError("HF_SECRET_NAME must be the exact label in Kaggle Add-ons > Secrets")
    try:
        if secret_client_factory is None:
            from kaggle_secrets import UserSecretsClient
            secret_client_factory = UserSecretsClient
        import time as _time
        for _attempt in range(3):  # the secrets service can refuse connections in API-started sessions
            try:
                value = secret_client_factory().get_secret(secret_name)
                break
            except ConnectionError:
                if _attempt == 2:
                    raise
                _time.sleep(10)
    except Exception as exc:
        # Never display exception text: upstream responses can contain credentials.
        error_type = type(exc).__name__
        raise RuntimeError(
            "Kaggle could not read the Hugging Face secret (" + error_type + "). "
            "Open this notebook in Edit mode > Add-ons > Secrets. Add/select the secret "
            "whose label exactly matches HF_SECRET_NAME and ENABLE its access for THIS notebook. "
            ""
            "Use a Hugging Face token with write access to " + HF_REPO_ID + ". "
            "Then Save Version > Save & Run All again. Do not paste the token into notebook code."
        ) from None
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError("The selected Kaggle secret is empty; update its value in Add-ons > Secrets")
    environ["HF_TOKEN"] = value.strip()
    return "Kaggle Secrets"

credential_source = load_hf_token(HF_SECRET_NAME)
print("Hugging Face credential loaded from", credential_source, "(value hidden)")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
if not AUDIT_ONLY or QUALITY_DEVICE == "cuda":
    try:
        gpu_list = subprocess.check_output(["nvidia-smi", "-L"], text=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        gpu_list = ""
    if "GPU" not in gpu_list:
        raise RuntimeError("Enable an NVIDIA GPU in Kaggle Settings; check remaining GPU quota")
    print(gpu_list.strip())
if not CLONE.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=CLONE, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=CLONE, check=True)
REPO = CLONE / PROJECT_FOLDER if (CLONE / PROJECT_FOLDER).is_dir() else CLONE
for name in ("setup_kaggle.py", "finetune_mixed.py", "stage2_data.py", "stage2_plan.py", "checkpoint_init.py", "urdutts_data.py", "audio_storage.py", "workflow_gate.py", "configs/mixed.json", "configs/stage2.json"):
    if not (REPO / name).is_file():
        raise RuntimeError(f"Missing {name}: push the mixed-data folder and check REPO_REF/PROJECT_FOLDER")
PROJECT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CLONE, text=True).strip()
print("Project commit (pin REPO_REF to this for resumes):", PROJECT_COMMIT)
early_check = subprocess.run([sys.executable, str(REPO / "check_hub.py"), "--repo", HF_REPO_ID], cwd=REPO)
if early_check.returncode == 2:
    raise RuntimeError("Hugging Face storage check failed; see redacted diagnostics above")
if early_check.returncode:
    print("Early Hub check unavailable in host Python; it will run in the installed training venv")

In [ ]:
# Dedicated venv inherits Kaggle CUDA wheels; installs project dependencies every fresh session.
ENV = Path("/kaggle/working/orpheus-mixed-env")
TRAIN_PYTHON = str(ENV / "bin" / "python")
setup_command = [sys.executable, "-u", str(REPO / "setup_kaggle.py"), "--venv", str(ENV)]
if not AUDIT_ONLY:
    setup_command.append("--gpu")
subprocess.run(setup_command, cwd=REPO, check=True)
subprocess.run([TRAIN_PYTHON, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=REPO, check=True)
subprocess.run(["df", "-h", "/kaggle/working"], check=True)

In [ ]:
# Generate every adapter's samples (same prompts and seeds), then transcribe with Whisper large-v3.
OUT = Path("/kaggle/working/pron-eval") / EVAL_NAME
command = [TRAIN_PYTHON, "-u", str(REPO / "eval_pronunciation.py"), "--output", str(OUT), "--repeats", str(REPEATS)]
for spec in ADAPTERS:
    command += ["--adapter", spec]
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
# Upload summary, transcripts and audio (adapter weights stay out of the upload).
from huggingface_hub import HfApi
HfApi().upload_folder(repo_id=HF_REPO_ID, folder_path=str(OUT), path_in_repo=f"evaluations/{EVAL_NAME}",
                      ignore_patterns=["adapters/**", "*.tokens.json"], commit_message=f"Pronunciation eval {EVAL_NAME}")
print(open(OUT / "summary.json").read())
